# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** Each entity (record set, field, column) is referenced by its `@id` field, per the Croissant schema.

In [ ]:
# List available record sets by @id
print("Available record sets (referenced by '@id'):")
for recset in metadata.record_sets:
    print(f"- @id: {recset.id_} | Name: {recset.name}")

# For each record set, list its fields by @id
for recset in metadata.record_sets:
    print(f"\nFields for record set '@id': {recset.id_}")
    for field in recset.fields:
        print(f"  - Field @id: {field.id_}, name: {field.name}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set, dynamically, referencing by @id
record_sets_ids = [recset.id_ for recset in metadata.record_sets]
dataframes = {}

for recset_id in record_sets_ids:
    records = list(dataset.records(record_set=recset_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"\n---- Columns for record set '@id': {recset_id} ----")
        print(df.columns.tolist())
        print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**For demonstration, we'll select the first record set with at least one numeric field, and show filtering, normalization, and grouping.**

In [ ]:
# Find a numeric field for demonstration
numeric_field_id = None
group_field_id = None
chosen_recset_id = None
for recset in metadata.record_sets:
    candidate_fields = [f for f in recset.fields if f.data_type in ('Number', 'Float', 'Integer')]
    if candidate_fields and recset.id_ in dataframes and not dataframes[recset.id_].empty:
        numeric_field_id = candidate_fields[0].id_  # Use the first available
        if len(candidate_fields) > 1:
            group_field_id = candidate_fields[1].id_
        else:
            if len(recset.fields) > 1:
                # Try second non-numeric field as group
                for f in recset.fields:
                    if f.data_type not in ('Number', 'Float', 'Integer'):
                        group_field_id = f.id_
                        break
        chosen_recset_id = recset.id_
        break

if numeric_field_id is not None:
    print(f"Selected record set: {chosen_recset_id}")
    print(f"Numeric field: {numeric_field_id}")
    if group_field_id:
        print(f"Group field: {group_field_id}")
else:
    print("No numeric fields found for EDA.")

# Run EDA if possible
if numeric_field_id is not None:
    df = dataframes[chosen_recset_id].copy()
    # Safely turn column labels into strings to match dictionary keys if needed
    colnames = [str(c) for c in df.columns]
    if numeric_field_id not in colnames:
        print(f"Numeric field '{numeric_field_id}' not found in data columns.")
    else:
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"],].head())
        # Group if possible
        if group_field_id and group_field_id in colnames:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. The following code will plot the distribution of the selected numeric field and, if a group field exists, the grouping mean.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id} in record set {chosen_recset_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # If grouping is possible, plot group means
    if group_field_id and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values().tail(10)
        plt.figure(figsize=(8,4))
        sns.barplot(y=group_means.index, x=group_means.values, orient='h')
        plt.title(f"Top 10 Means of {numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(numeric_field_id + ' (mean)')
        plt.ylabel(group_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset was loaded and explored using the `mlcroissant` library.
- All entities (record sets, fields, etc.) were referenced by their Croissant `@id` identifiers.
- We identified available record sets, fields, and demonstrated EDA on numeric fields, including filtering and normalization.
- Data visualizations highlighted numeric field distributions and groupwise means where possible.
- For advanced applications, further analyses can be repeated for other fields/record sets using their `@id`s.